# 订单簿：先预测，再运行

本实验用 Python 3 标准库，运行所有单元前无需下载行情或连接交易账户。整数数量和价格均为教学单位，不是 BTC 产品参数。模型仅实现 GTC、被动单价格执行，省略持久化、风险、特殊订单和生产精度。可从新内核按顺序运行；不依赖同目录文件。


In [ ]:
"""Tiny price-time-priority teaching simulator.

This is deliberately single-threaded and in-memory. It is not a production
matching engine: it omits validation, integer tick/lot units, persistence,
risk checks, self-trade prevention, special time-in-force rules, and market
orders.
"""

from dataclasses import dataclass
from itertools import count

_sequence = count()


@dataclass
class Order:
    order_id: str
    side: str
    price: int
    qty: int
    sequence: int


class Book:
    def __init__(self):
        self.bids = []
        self.asks = []
        self.trades = []

    def add_limit(self, order_id, side, price, qty):
        if side not in {"BUY", "SELL"} or qty <= 0 or price <= 0:
            raise ValueError("side must be BUY/SELL and price/qty must be positive")

        incoming = Order(order_id, side, price, qty, next(_sequence))
        opposite = self.asks if side == "BUY" else self.bids
        resting_side = self.bids if side == "BUY" else self.asks

        while incoming.qty and opposite:
            resting = opposite[0]
            crosses = price >= resting.price if side == "BUY" else price <= resting.price
            if not crosses:
                break

            fill_qty = min(incoming.qty, resting.qty)
            self.trades.append((incoming.order_id, resting.order_id, resting.price, fill_qty))
            incoming.qty -= fill_qty
            resting.qty -= fill_qty
            if resting.qty == 0:
                opposite.pop(0)

        if incoming.qty:
            resting_side.append(incoming)
            if side == "BUY":
                resting_side.sort(key=lambda o: (-o.price, o.sequence))
            else:
                resting_side.sort(key=lambda o: (o.price, o.sequence))

        return incoming.qty

    def show(self):
        print("Trades (taker, maker, price, qty):")
        for trade in self.trades:
            print(" ", trade)
        print("Bids:", [(o.price, o.qty, o.order_id) for o in self.bids])
        print("Asks:", [(o.price, o.qty, o.order_id) for o in self.asks])




## 第一轮：同价排队

先写出预测：Alice 卖2@100、Bo卖1@101、Chen卖3@100；Dana买4@100。谁先成交？Chen还剩多少？


In [ ]:
book = Book()
book.add_limit("Alice", "SELL", 100, 2)
book.add_limit("Bo", "SELL", 101, 1)
book.add_limit("Chen", "SELL", 100, 3)
book.add_limit("Dana", "BUY", 100, 4)
book.show()


解释输出：Dana与Alice成交2、与Chen成交2，均按100。卖盘剩Chen 1@100和Bo 1@101。为什么Bo没有先于Chen成交？

## 第二轮：只改变同价到达顺序

预测将Chen移到Alice之前会改变哪几笔成交，什么仍然不变。下一格新建独立订单簿。


In [ ]:
book = Book()
book.add_limit("Chen", "SELL", 100, 3)
book.add_limit("Bo", "SELL", 101, 1)
book.add_limit("Alice", "SELL", 100, 2)
book.add_limit("Dana", "BUY", 100, 4)
book.show()


结果应为Chen 3、Alice 1；更优价格仍先于101。

## 第三轮：吃单余量成为挂单

恢复原卖簿，Dana买6@100。先预测余量和角色；再加入Eve卖1@99，预测执行价是否为99。


In [ ]:
book = Book()
for oid, price, qty in [("Alice",100,2),("Bo",101,1),("Chen",100,3)]:
    book.add_limit(oid, "SELL", price, qty)
book.add_limit("Dana", "BUY", 100, 6)
book.show()
book.add_limit("Eve", "SELL", 99, 1)
book.show()


最后一笔按被动买单100执行，Eve是taker、Dana是maker；Bo仍挂在101。

## 重建

关闭输出后写出匹配循环：最优对手价、限价条件、min余量、更新删除、继续匹配、GTC余量入队。再手算IOC/FOK/Post Only，解释为何不能只改余量入队这一行实现所有策略。
